# Person 1 — Colab Pro GPU Training Notebook (Data, Tokenizers, Baseline & Legal-BERT)

**How to use this notebook on Google Colab Pro:**
1. Open [Google Colab](https://colab.research.google.com/), click **Upload**, and select this file (`train_colab_pro.ipynb`).
2. In the top menu, go to **Runtime -> Change runtime type** and select **GPU (A100 or L4 / T4)**.
3. Click **Runtime -> Run all**.
4. At the end of the notebook, it will automatically package and download `outputs.zip` containing:
   - `outputs/class_distribution.csv` & `outputs/tokenizer_comparison.json`
   - `outputs/tfidf_val_logits.npy` & `outputs/tfidf_test_logits.npy` (Baseline predictions)
   - `outputs/legal_bert_val_logits.npy` & `outputs/legal_bert_test_logits.npy` (Fine-tuned Legal-BERT predictions)
   - `outputs/val_labels.npy`, `outputs/test_labels.npy`, `outputs/label_names.json`
5. Unzip `outputs.zip` into your local project folder (`outputs/`) and push everything to GitHub for your teammates!

In [ ]:
# Step 0: Check Colab Pro GPU & Install Dependencies
!nvidia-smi
!pip install -q datasets transformers accelerate scikit-learn pandas numpy tqdm

In [ ]:
# Step 1: Load LEDGAR Dataset & Compute Class Imbalance Table
import json
from pathlib import Path
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer

OUTPUTS_DIR = Path("outputs")
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

ledgar = load_dataset("coastalcph/lex_glue", "ledgar")
label_names = ledgar["train"].features["label"].names
num_classes = len(label_names)
print(ledgar)

train_counts = np.bincount(ledgar["train"]["label"], minlength=num_classes)
val_counts = np.bincount(ledgar["validation"]["label"], minlength=num_classes)
test_counts = np.bincount(ledgar["test"]["label"], minlength=num_classes)

class_df = pd.DataFrame([
    {
        "class_id": i,
        "class_name": name,
        "train_count": int(train_counts[i]),
        "val_count": int(val_counts[i]),
        "test_count": int(test_counts[i]),
    }
    for i, name in enumerate(label_names)
]).sort_values("train_count", ascending=False).reset_index(drop=True)

class_df.to_csv(OUTPUTS_DIR / "class_distribution.csv", index=False)
print("Most frequent class:", class_df.iloc[0].to_dict())
print("Least frequent class:", class_df.iloc[-1].to_dict())
print("Number of classes with < 10 test examples:", int((class_df["test_count"] < 10).sum()))

In [ ]:
# Step 2: Compare Tokenizers (Legal-BERT vs. BERT-base vs. CamemBERT)
# Proves why we use Legal-BERT instead of CamemBERT on English SEC contracts
sample_texts = list(ledgar["train"]["text"][:500])
word_counts = np.array([max(1, len(t.split())) for t in sample_texts])

tokenizer_stats = {}
for model_name in ["nlpaueb/legal-bert-base-uncased", "bert-base-uncased", "camembert-base"]:
    tok = AutoTokenizer.from_pretrained(model_name)
    enc = tok(sample_texts, truncation=False)
    tok_lens = np.array([len(ids) for ids in enc["input_ids"]])
    tokenizer_stats[model_name] = {
        "mean_tokens_per_word": round(float(np.mean(tok_lens / word_counts)), 3),
        "median_token_length": float(np.median(tok_lens)),
        "p95_token_length": float(np.percentile(tok_lens, 95)),
        "pct_truncated_at_256": round(float(100.0 * np.mean(tok_lens > 256)), 2),
        "pct_truncated_at_512": round(float(100.0 * np.mean(tok_lens > 512)), 2),
    }

with open(OUTPUTS_DIR / "tokenizer_comparison.json", "w") as f:
    json.dump(tokenizer_stats, f, indent=2)

print(json.dumps(tokenizer_stats, indent=2))

In [ ]:
# Step 3: Train Fast TF-IDF + Logistic Regression Baseline & Save Logits
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

train_texts = list(ledgar["train"]["text"])
train_labels = np.array(ledgar["train"]["label"], dtype=np.int64)
val_texts = list(ledgar["validation"]["text"])
val_labels = np.array(ledgar["validation"]["label"], dtype=np.int64)
test_texts = list(ledgar["test"]["text"])
test_labels = np.array(ledgar["test"]["label"], dtype=np.int64)

print("Fitting TF-IDF + Logistic Regression baseline on 60,000 clauses...")
vectorizer = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), sublinear_tf=True, min_df=2)
X_train = vectorizer.fit_transform(train_texts)
X_val = vectorizer.transform(val_texts)
X_test = vectorizer.transform(test_texts)

clf = LogisticRegression(C=2.0, max_iter=300, random_state=42)
clf.fit(X_train, train_labels)

tfidf_val_logits = clf.decision_function(X_val).astype(np.float32)
tfidf_test_logits = clf.decision_function(X_test).astype(np.float32)

tfidf_val_acc = float(np.mean(np.argmax(tfidf_val_logits, axis=1) == val_labels))
tfidf_test_acc = float(np.mean(np.argmax(tfidf_test_logits, axis=1) == test_labels))
print(f"TF-IDF Baseline -> Val Accuracy: {tfidf_val_acc:.4f} | Test Accuracy: {tfidf_test_acc:.4f}")

np.save(OUTPUTS_DIR / "tfidf_val_logits.npy", tfidf_val_logits)
np.save(OUTPUTS_DIR / "tfidf_test_logits.npy", tfidf_test_logits)
np.save(OUTPUTS_DIR / "val_labels.npy", val_labels)
np.save(OUTPUTS_DIR / "test_labels.npy", test_labels)
with open(OUTPUTS_DIR / "label_names.json", "w") as f:
    json.dump(label_names, f, indent=2)

In [ ]:
# Step 4: Fine-Tune Legal-BERT on Colab Pro GPU & Export Validation/Test Logits
import time
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import (
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_cosine_schedule_with_warmup,
)

MODEL_NAME = "nlpaueb/legal-bert-base-uncased"
MAX_LEN = 256
BATCH_SIZE = 64  # Fast on Colab Pro A100/L4 (reduce to 32 if using T4)
EPOCHS = 3
LR = 3e-5
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training on device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Pre-tokenize splits once in memory for maximum GPU utilization
def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

tokenized_ledgar = ledgar.map(tokenize_batch, batched=True, remove_columns=["text"])
tokenized_ledgar = tokenized_ledgar.rename_column("label", "labels")

collator = DataCollatorWithPadding(tokenizer=tokenizer)
train_loader = DataLoader(tokenized_ledgar["train"], batch_size=BATCH_SIZE, shuffle=True, collate_fn=collator)
val_loader = DataLoader(tokenized_ledgar["validation"], batch_size=128, shuffle=False, collate_fn=collator)
test_loader = DataLoader(tokenized_ledgar["test"], batch_size=128, shuffle=False, collate_fn=collator)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_classes).to(device)
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps
)
criterion = nn.CrossEntropyLoss()
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

@torch.no_grad()
def eval_loader(m, loader):
    m.eval()
    logits_list, labels_list = [], []
    total_loss, total_n = 0.0, 0
    for batch in loader:
        labels = batch.pop("labels").to(device)
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
            out = m(**batch)
            loss = criterion(out.logits, labels)
        bs = labels.size(0)
        total_loss += float(loss.item()) * bs
        total_n += bs
        logits_list.append(out.logits.detach().cpu().numpy().astype(np.float32))
        labels_list.append(labels.detach().cpu().numpy().astype(np.int64))
    return np.concatenate(logits_list), np.concatenate(labels_list), total_loss / max(1, total_n)

history = []
best_val_acc = -1.0
best_val_logits = None

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    model.train()
    running_loss, seen = 0.0, 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}")
    for batch in pbar:
        optimizer.zero_grad(set_to_none=True)
        labels = batch.pop("labels").to(device)
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
            out = model(**batch)
            loss = criterion(out.logits, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        bs = labels.size(0)
        running_loss += float(loss.item()) * bs
        seen += bs
        pbar.set_postfix({"train_loss": f"{running_loss / seen:.4f}"})

    val_logits, val_y, val_loss = eval_loader(model, val_loader)
    val_acc = float(np.mean(np.argmax(val_logits, axis=1) == val_y))
    rec = {
        "epoch": epoch,
        "train_loss": round(running_loss / seen, 4),
        "val_loss": round(val_loss, 4),
        "val_accuracy": round(val_acc, 4),
        "elapsed_sec": round(time.time() - t0, 1),
    }
    history.append(rec)
    print(rec)
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_val_logits = val_logits
        model.save_pretrained("best_legal_bert")
        tokenizer.save_pretrained("best_legal_bert")

best_model = AutoModelForSequenceClassification.from_pretrained("best_legal_bert").to(device)
test_logits, test_y, test_loss = eval_loader(best_model, test_loader)
test_acc = float(np.mean(np.argmax(test_logits, axis=1) == test_y))
print(f"Final Legal-BERT Test Accuracy (Micro-F1): {test_acc:.4f} (Test Loss: {test_loss:.4f})")

np.save(OUTPUTS_DIR / "legal_bert_val_logits.npy", best_val_logits)
np.save(OUTPUTS_DIR / "legal_bert_test_logits.npy", test_logits)
with open(OUTPUTS_DIR / "training_history.json", "w") as f:
    json.dump({"model": MODEL_NAME, "test_accuracy": round(test_acc, 4), "history": history}, f, indent=2)

In [ ]:
# Step 5: Zip and Download `outputs.zip` to Your Mac
import shutil
shutil.make_archive("outputs", "zip", root_dir=".", base_dir="outputs")
print("Created outputs.zip!")

try:
    from google.colab import files
    files.download("outputs.zip")
except ImportError:
    print("Not running in Google Colab; outputs are saved in ./outputs/")